In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns



## Hypothesis

1) Plants are disproportionately located in areas with higher % People of Color
2) Plants are disproportionately located in areas with higher % Low Income
3) Plants are disproportionately located in areas with higher % Less Than High School Education
4) Plants are disproportionately located in areas with higher % Unemployment Rate
5) Plants are disproportionately located in areas with higher % Limited English Speaking

In [3]:
counties = pd.read_csv('data/mh_analysis_ready.csv', index_col=0)
counties = counties.drop(columns=['Plant state abbreviation_x', 'Limited Life Expectancy (%)', 'Plant primary fuel category_x', 'Plant annual net generation (MWh)'])
counties = counties.dropna()
counties

,County FIPS,Plant state abbreviation,Plant county name,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),has_plant
2,1001,AL,Autauga County,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,0
3,1003,AL,Baldwin County,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,0
4,1005,AL,Barbour County,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,0
5,1007,AL,Bibb County,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,0
6,1009,AL,Blount County,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,0
...,...,...,...,...,...,...,...,...,...,...
8510,56009,WY,Converse,289.0,11.000000,24.000000,6.000000,0.000000,0.000000,1
8512,56005,WY,Campbell,1861.0,16.000000,29.000000,8.000000,0.000000,2.000000,1
8513,56005,WY,Campbell,1638.0,15.000000,29.000000,7.000000,0.000000,2.000000,1
8514,56005,WY,Campbell,1649.0,15.000000,29.000000,7.000000,0.000000,2.000000,1


In [4]:
def difference_of_means(table, group_label, numerical_col):
    
    series = table.groupby('Shuffled Label').mean().loc[:, numerical_col]
   
    return abs(series.iloc[1] - series.iloc[0])

In [5]:
def one_simulated_difference_of_means(numerical_col, binary_col, i):

    shuffled_table = counties.copy()
    
    shuffled_labels = counties.sample(replace=False, frac = 1, random_state = i)[binary_col]
    shuf = shuffled_labels.to_numpy()
    shuffled_table['Shuffled Label'] = shuf
    selected_shuf = shuffled_table.loc[:, (numerical_col, 'Shuffled Label')]
    
    return difference_of_means(selected_shuf, 'Shuffled Label', numerical_col)   

In [6]:
def avg_difference_in_means(numerical_col, binary_col= 'has_plant'):
    """
   The function computes the p-value for a test of the following hypothesis test:
        H0 : There is no difference in the average value of numerical_col between the two
            groups specified in binary_col.
        H1 : The average value of numerical_col is different for the two groups specified
            in binary_col
    inputs
        numerical_col: a numerical column name
        binary_col: a binary column name
    """
    selected = counties.loc[:, (numerical_col, binary_col)]
    series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col]
    observed_difference = abs(series_obs.iloc[1] - series_obs.iloc[0]) 

    differences = []

    repetitions = 100
    for i in np.arange(repetitions):
        new_difference = one_simulated_difference_of_means(numerical_col, binary_col, i)
        differences = np.append(differences, new_difference)                               

    empirical_p = np.count_nonzero(differences >= observed_difference) / repetitions #how many samples have as extreme of a difference?

  
    #print(f' observed dif: {observed_difference}, empirical p: {empirical_p}')
    return empirical_p
    

In [7]:
numerical_cols = ['People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)', 'Limited English Speaking (%)' ]
binary_cols = ['has_plant']
pvals = {}
for i in numerical_cols:
    for j in binary_cols:
        pvals[f'{i} and {j}'] = avg_difference_in_means(i, j)

pvals

{'People of Color (%) and has_plant': 0.0,
 'Low Income (%) and has_plant': 0.0,
 'Less Than High School Education (%) and has_plant': 0.77,
 'Unemployment Rate (%) and has_plant': 0.3,
 'Limited English Speaking (%) and has_plant': 0.04}

In [8]:
fwer = 0.05
num_tests = len(numerical_cols) * len(binary_cols)
fwer_threshold = fwer / num_tests
print(f'FWER threshold: {fwer_threshold}')
reject_null = [x for x in list(pvals.keys()) if pvals[x] <= fwer_threshold]
#list(pvals.values())
reject_null

FWER threshold: 0.01


['People of Color (%) and has_plant', 'Low Income (%) and has_plant']

In [9]:

#B-H procedure:  
p_sorted = sorted(list(pvals.values()))

m = len(p_sorted)  
k = np.arange(1, m+1)  # index of each test in sorted order
desired_fdr = 0.05
compare = (desired_fdr * k) /m
below_than = p_sorted < compare
cols = {'k' : k, 'p-vals' : p_sorted, 'compare': compare, 'below than' : below_than}

ps = pd.DataFrame(cols)
ps


,k,p-vals,compare,below than
0,1,0.00,0.01,True
1,2,0.00,0.02,True
2,3,0.04,0.03,False
3,4,0.30,0.04,False
4,5,0.77,0.05,False


## Methods

– Describe the hypotheses that you’ll be testing using your dataset, and explain why it
makes sense to test many hypotheses instead of just one to answer your question.

We will be testing the following hypotheses: 

Plants are disproportionately located in areas with higher % People of Color

Plants are disproportionately located in areas with higher % Low Income

Plants are disproportionately located in areas with higher % Less Than High School Education

Plants are disproportionately located in areas with higher % Unemployment Rate

Plants are disproportionately located in areas with higher % Limited English Speaking

We are interested in 


– For at least one of the tests, describe a specific alternative hypothesis, and compute
the power of the test you plan to use. You must be very clear in specifying the
alternative hypothesis and why you chose the values you did.

– Describe how you’ll be testing each hypothesis (A/B test, correlation/association, etc.),
and justify your choice.

We will be using difference-in-means testing


– Describe at least two different ways you’ll correct for multiple hypothesis tests, and explain
the error rates being controlled.

– Compare and contrast FWER control and FDR control for your tests: specifically, use
one method that controls FWER and one that controls FDR, and compare the number of
discoveries made by each. Explain which is more appropriate for your research question.

We will use the Bonferroni correction and the Benjamini-Hochberg procedure to control for FWER and FDR, respectively. 




## Results 

– Summarize and interpret the results from the hypothesis tests themselves.
